# BRFSS 2015 — Colab research runner
Default: **NEW_PROTOCOL**, a new internal-validation study, not a restoration or independent validation of old results. **LEGACY_RESUME** fails unless the complete immutable legacy bundle passes its audit. Synthetic mode validates software only. Final-test selection is forbidden. Source-only GitHub downloads support NEW_PROTOCOL.

Set optional environment variables `BRFSS_REPO`, `BRFSS_DATA`, `BRFSS_CODEBOOK`, `BRFSS_PROTOCOL`, `BRFSS_PROFILE`, `BRFSS_OUT` before Run all. Python 3.10–3.12 is supported; other versions require dependency revalidation. If setup changes imported binary packages, restart the session and Run all again.

In [1]:
import os
import sys
import subprocess
from pathlib import Path

SYNTHETIC = os.environ.get("BRFSS_SYNTHETIC", "0") == "1"
if not (3, 10) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError(f"Python {sys.version.split()[0]} is outside tested 3.10–3.12. Select a compatible Colab runtime; dependency validation is required for newer Python.")
if not SYNTHETIC:
    from google.colab import drive
    drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive")
explicit = os.environ.get("BRFSS_REPO")
candidates = [Path(explicit)] if explicit else [
    Path.cwd(), Path.cwd().parent,
    drive_root / "PPTK-main (1)/PPTK-main/BRFSS2015-Statistical-Methods",
    drive_root / "BRFSS2015-Statistical-Methods",
    drive_root / "BRFSS2015_Statistical_Methods_AZ/BRFSS2015-Statistical-Methods",
]
if not explicit and drive_root.is_dir():
    candidates += sorted(drive_root.glob("PPTK-main*/PPTK-main/BRFSS2015-Statistical-Methods"))
REPO = next((p.resolve() for p in candidates if (p / "pyproject.toml").is_file()), None)
if REPO is None:
    raise FileNotFoundError(f"Set BRFSS_REPO to the project folder. Checked: {candidates}")
required = ["src/data/ingest.py", "src/data/outcomes.py", "src/data/synthetic.py", "scripts/colab_runtime.py", "configs/project.yaml"]
missing = [name for name in required if not (REPO / name).is_file()]
if missing:
    raise FileNotFoundError(f"Incomplete source checkout: {missing}. Update the repository; no missing files will be synthesized.")
os.chdir(REPO)
sys.path.insert(0, str(REPO)) if str(REPO) not in sys.path else None
print("Repository:", REPO, "Python:", sys.version.split()[0])

Repository: /tmp/brfss-source-only-checkout Python: 3.12.14


In [2]:
if not SYNTHETIC:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
    if os.environ.get("BRFSS_INSTALL_OPTIONAL", "1") == "1":
        for requirement in ["requirements-boosters.txt", "requirements-research-extensions.txt"]:
            subprocess.run([sys.executable, "-m", "pip", "install", "-r", requirement], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)
from scripts.verify_environment import verify
versions = verify()
# Import every project module before accessing data.
import importlib
import pkgutil
for package_name in ["src", "src_extensions"]:
    package = importlib.import_module(package_name)
    for module in pkgutil.walk_packages(package.__path__, package_name + "."):
        importlib.import_module(module.name)

No broken requirements found.

{
  "python": "3.12.14",
  "numpy": "2.2.6",
  "pandas": "2.2.3",
  "scipy": "1.15.3",
  "scikit-learn": "1.6.1",
  "statsmodels": "0.14.4",
  "imbalanced-learn": "0.13.0",
  "matplotlib": "3.10.1",
  "seaborn": "0.13.2",
  "PyYAML": "6.0.2",
  "pyreadstat": "1.2.8",
  "nbformat": "5.10.4",
  "nbclient": "0.10.2",
  "ipykernel": "6.29.5",
  "pytest": "8.3.5",
  "ruff": "0.11.13",
  "requests": "2.32.3",
  "pypdf": "5.5.0"
}


In [3]:
from scripts.colab_runtime import execute, locate_data
PROTOCOL = os.environ.get("BRFSS_PROTOCOL", "NEW_PROTOCOL")
PROFILE = os.environ.get("BRFSS_PROFILE", "quick").lower()
MODE = os.environ.get("BRFSS_MODE", "binary")
SEED = int(os.environ.get("BRFSS_SEED", "20261107"))
OUT = Path(os.environ.get("BRFSS_OUT", str(REPO / "runs" / ("new_protocol_synthetic" if SYNTHETIC else "new_protocol_2015"))))
DATA = None if SYNTHETIC else str(locate_data(os.environ.get("BRFSS_DATA")))
CODEBOOK = os.environ.get("BRFSS_CODEBOOK")
if not SYNTHETIC and not CODEBOOK:
    from src.data.ingest import discover
    CODEBOOK = str(discover(directories=[Path(DATA).parent], filename="codebook15_llcp.pdf"))
execute(protocol=PROTOCOL, synthetic=SYNTHETIC, data=DATA, codebook=CODEBOOK,
        out=OUT, profile=PROFILE, mode=MODE, seed=SEED)

2026-11-06 16:02:57,390 INFO RESUME audit


2026-11-06 16:02:57,392 INFO RESUME cohort


2026-11-06 16:02:57,392 INFO RESUME descriptive


2026-11-06 16:02:57,393 INFO RESUME inference


2026-11-06 16:02:57,398 INFO RESUME split


2026-11-06 16:02:57,401 INFO RESUME develop


2026-11-06 16:02:57,402 INFO RESUME evaluate


2026-11-06 16:02:57,403 INFO RESUME report


PosixPath('/tmp/brfss-source-only-output')

In [4]:
from src.utils.io import read_json
from IPython.display import display, Markdown
if PROTOCOL == "NEW_PROTOCOL":
    ledger = read_json(OUT / "test_access.json")
    assert ledger["state"] == "COMPLETE" and ledger["evaluation_count"] == 1
    display(Markdown((OUT / "final_report.md").read_text()))
    print(read_json(OUT / "protocol.json"))
else:
    print(read_json(OUT / "integrity.json"))
print("Outputs:", OUT)
print("SYNTHETIC SOFTWARE VALIDATION ONLY" if SYNTHETIC else "Real-data run; review audit and protocol scope before interpreting results.")
print("COLAB_RUNTIME_NOT_VERIFIED by repository CI; an executed notebook in your actual Colab session is separate evidence.")

# Synthetic validation report — not epidemiological evidence

## Abstract

This cross-sectional analysis investigates self-reported diabetes associations and fixed-protocol predictive classification. Estimates describe associations, not causal effects or future incident disease.

## Research questions and prespecified analysis

RQ1: adjusted associations using complex survey design. RQ2: training-only feature, imbalance and model comparisons. RQ3: balanced accuracy, sensitivity, calibration and discrimination versus a same-cohort logistic baseline. RQ4: subgroup and outcome/missingness sensitivity. Full estimands and hypotheses are in docs/RESEARCH_PROTOCOL.md.

## Data and cohort

Source records: 960. Eligible: 871. Complete cases: 662. Design strata: 120. Profile: quick. Source fingerprints and exact schema are in audit.json.

## Methods

CDC core DIABETE3=1 versus 3 is the primary contrast; pregnancy-only and borderline responses are excluded. Optional prediabetes ascertainment is not assumed complete. Survey point estimates use _LLCPWT and variance uses stratified _PSU score totals within _STSTR, without a finite population correction. Complete-case inference and median/mode sensitivity target different missing-data assumptions. Preprocessing and feature selection are fitted inside training folds. The final test set is accessed in one frozen-model batch.

## Weighted prevalence

| class | estimate | lower | upper | n |
| --- | --- | --- | --- | --- |
| 0.0 | 0.86023 | 0.8336 | 0.8832 | 871 |
| 1.0 | 0.13977 | 0.1168 | 0.1664 | 871 |

Weighted diabetes prevalence within this defined analysis domain was 13.98% (95% design CI 11.68% to 16.64%). This domain excludes unknown outcomes and may exclude borderline/pregnancy-only responses; it is not an unconditional whole-population estimate.

## Statistical associations

Adjusted OR, confidence limits, design-based p-values, and exploratory BH corrections: adjusted_or.csv. Multinomial RRR: multinomial_rrr.csv when completed. Univariate results: univariate.csv. Clinical interpretation must consider coding references, precision, multiplicity, selection, and reverse causation.

Prespecified illustrative contrasts are below; the complete coefficient table accompanies this report. OR/RRR compare coded levels with the first codebook category, holding included covariates fixed. BMI is per 10 kg/m²; no coefficient is a causal effect.

| term | OR | lower | upper | p |
| --- | --- | --- | --- | --- |
| _AGEG5YR_13 | 2.21134 | 0.6492 | 7.53241 | 0.20357 |
| SEX_2 | 1.0418 | 0.61261 | 1.77168 | 0.87949 |
| _BMI5_per_10 | 2.03451 | 1.16385 | 3.5565 | 0.01286 |
| INCOME2_8 | 0.68377 | 0.22568 | 2.07169 | 0.50031 |
| BPHIGH4_3 | 1.98819 | 0.95013 | 4.16036 | 0.06801 |
| TOLDHI2_2 | 1.25246 | 0.74569 | 2.10363 | 0.39364 |
| GENHLTH_5 | 0.63224 | 0.25793 | 1.54973 | 0.31507 |

## Model development

Development scores guided selection and threshold tuning; they are optimistic for the selected strategy. They are not final test performance. All candidates and skipped dependencies are in model_comparison.csv.

| model | selector | k | imbalance | cv_mean | threshold |
| --- | --- | --- | --- | --- | --- |
| extra_trees | original | 10 | none | 0.52483 | 0.2 |
| decision_tree | original | 10 | none | 0.5247 | 0.2 |
| logistic | original | 10 | smote | 0.52291 | 0.8 |
| logistic | tree | 10 | none | 0.52066 | 0.2 |
| xgboost | original | 10 | none | 0.51929 | 0.2 |
| random_forest | original | 10 | none | 0.5176 | 0.2 |
| logistic | original | 10 | random_under | 0.51749 | 0.5 |
| feature_ensemble | original | 10 | none | 0.51555 | 0.3 |
| logistic | original | 10 | class_weight | 0.51067 | 0.4 |
| gradient_boosting | original | 10 | none | 0.51043 | 0.5 |
| lightgbm | original | 10 | none | 0.5099 | 0.3 |
| soft_voting | original | 10 | none | 0.50392 | 0.5 |
| logistic | l1 | 10 | none | 0.50307 | 0.4 |
| logistic | mutual_info | 10 | none | 0.5029 | 0.3 |
| stacking | original | 10 | none | 0.50238 | 0.2 |

## Predictive results

| metric | selected | baseline | CI_lower | CI_upper |
| --- | --- | --- | --- | --- |
| accuracy | 0.73714 | 0.84 | 0.677 | 0.79729 |
| balanced_accuracy | 0.51226 | 0.49329 | 0.42627 | 0.59678 |
| sensitivity | 0.19231 | 0.0 | 0.05402 | 0.38014 |
| specificity | 0.83221 | 0.98658 | 0.7837 | 0.88282 |
| precision | 0.16667 | 0.0 | 0.03075 | 0.28359 |
| f1 | 0.17857 | 0.0 | 0.03919 | 0.30813 |
| roc_auc | 0.55679 | 0.59293 | 0.46711 | 0.67044 |
| pr_auc | 0.22407 | 0.19396 | 0.13104 | 0.38107 |
| brier | 0.12679 | 0.13438 | 0.09872 | 0.15944 |
| log_loss | 0.41986 | 0.43763 | 0.34747 | 0.50345 |

The frozen model was evaluated on 175 test respondents once. Balanced-accuracy difference versus the fixed same-cohort logistic comparator was 0.0190 (paired conditional bootstrap 95% interval -0.0651 to 0.1034). Intervals describe test respondents conditional on fixed fitted models; they do not include survey population or model-selection uncertainty. See final_metrics.json for all per-class errors, comparisons and weighted point estimates.

## Research gaps and interpretation

The gap matrix distinguishes inspected evidence from hypotheses. Chang et al. (2022), DOI 10.1016/j.health.2022.100118, reported 82.26% accuracy; our cohort/outcome and evaluation may differ. A higher point estimate here does not establish improvement over that publication. The implemented comparison estimates improvement over a prespecified logistic model on identical test respondents. Feature importance measures predictive utility, not causal contribution. Prediction, methodological validity, interpretability, and epidemiological evidence are separate contributions.

## Sensitivity and generalizability

See sensitivity_or.csv, borderline_sensitivity_or.csv, multinomial_rrr.csv, and subgroups.csv. Missing outputs are unavailable rather than negative findings. Core-question borderline status is an imperfect three-category self-report; module analyses require geographic/version eligibility and appropriate weights. External or temporal validation has not been performed.

## Limitations

Self-reported cross-sectional data cannot distinguish all diabetes types, verify biological disease, or determine temporal direction. Complete-case inference may be selected. Deterministic imputation sensitivity is not multiple imputation. Calibrated weights are treated as fixed in the with-replacement linearization. Predictive bootstrap intervals are conditional respondent intervals, not full survey population intervals or uncertainty over model selection. Small demographic groups may be unstable. Quick mode caps training size and is a computational validation rather than a full experimental study.

## Reproducibility

Source hashes, code/config signature, package versions, split IDs/checksum, CV indices, experiment history, frozen model checksum, test-access ledger and atomic checkpoints accompany this report. See execution_status.json; actual Google Colab execution remains unverified.

## Conclusion

All numerical results in this report are synthetic software validation. They provide no evidence about diabetes prevalence, real-world associations, or predictive benefit.

## References

CDC BRFSS 2015 data and codebook: https://www.cdc.gov/brfss/annual_data/annual_2015.html

Chang V, Ganatra MA, Hall K, Golightly L, Xu Q (2022). An assessment of machine learning models and algorithms for early prediction and diagnosis of diabetes using health indicators. Healthcare Analytics 2:100118. https://doi.org/10.1016/j.health.2022.100118


{'signature': '87b4869664b6f9f36020784301847d0e18f8bdec4482345721720018146b44c4', 'protocol': 'NEW_PROTOCOL', 'seed': 20261107, 'profile': 'quick', 'mode': 'binary', 'synthetic': True, 'primary_metric': 'training CV balanced_accuracy', 'test_policy': 'one frozen selected model and prespecified baseline batch; no test-based selection', 'historical_test_overlap': 'UNKNOWN: historical split not used; not independent validation of previously published models', 'claim_scope': 'new within-run split only; cross-sectional internal validation'}
Outputs: /tmp/brfss-source-only-output
SYNTHETIC SOFTWARE VALIDATION ONLY
COLAB_RUNTIME_NOT_VERIFIED by repository CI; an executed notebook in your actual Colab session is separate evidence.
